# 2.2 编程题
## 手动实现支持步幅和填充的二维最大池化前向传播函数（使用NumPy）.

In [1]:
import numpy as np

def max_pool2d(input, kernel_size, stride, padding):
    """
    二维最大池化前向传播。
    
    参数:
        input: numpy数组，形状 (N, C, H, W)
        kernel_size: int 或 tuple (kh, kw)
        stride: int 或 tuple (sh, sw)
        padding: int 或 tuple (ph, pw)
    
    返回:
        output: numpy数组，形状 (N, C, H_out, W_out)
    """
    # 统一为元组格式
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    N, C, H, W = input.shape
    # 计算输出尺寸
    H_out = (H + 2*ph - kh) // sh + 1
    W_out = (W + 2*pw - kw) // sw + 1
    
    # 对输入进行填充
    padded = np.pad(input, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=0)
    
    output = np.zeros((N, C, H_out, W_out))
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    h_start = i * sh
                    h_end = h_start + kh
                    w_start = j * sw
                    w_end = w_start + kw
                    window = padded[n, c, h_start:h_end, w_start:w_end]
                    output[n, c, i, j] = np.max(window)
    return output

# 测试示例
if __name__ == "__main__":
    x = np.random.randn(2, 3, 32, 32)  # 批量2，通道3，高宽32
    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("输入形状:", x.shape)
    print("输出形状:", out.shape)  # 应为 (2,3,16,16)

输入形状: (2, 3, 32, 32)
输出形状: (2, 3, 16, 16)


# 3.2 编程题
## 使用PyTorch定义NiN块。

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 示例：输入通道3，输出通道16，3x3卷积，步幅1，填充1
block = NiNBlock(3, 16, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print("NiN块输出形状:", out.shape)  # torch.Size([1, 16, 32, 32])

NiN块输出形状: torch.Size([1, 16, 32, 32])


# 4.2 编程题
## 自定义残差块Residual。

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.use_1x1conv = use_1x1conv
        if use_1x1conv:
            self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
            self.bn_shortcut = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.use_1x1conv:
            identity = self.conv_shortcut(identity)
            identity = self.bn_shortcut(identity)
        
        out += identity
        out = self.relu(out)
        return out

# 测试
res_block = Residual(3, 16, use_1x1conv=True, stride=2)
x = torch.randn(1, 3, 32, 32)
out = res_block(x)
print("残差块输出形状:", out.shape)  # torch.Size([1, 16, 16, 16])

残差块输出形状: torch.Size([1, 16, 16, 16])


## 5.1 微调理论问题

1. **为什么底层用小学习率，顶层用大学习率？**  
   底层特征（如边缘、纹理）具有通用性，预训练模型已学到良好表示，使用小学习率可以避免破坏这些通用特征；顶层输出层与具体任务相关，随机初始化后需要较大学习率快速适应新数据集。

2. **目标数据集小且与源数据集相似时的微调策略**  
   应冻结大部分底层卷积层，仅微调最后几层或只训练新添加的分类器。同时使用较小的学习率（如 \(10^{-4}\) 或更低），并采用数据增广、Dropout、权重衰减等正则化手段防止过拟合。

# 5.2 编程题
## 使用torchvision.transforms创建图像增广管道。

In [ ]:
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),  # 随机裁剪并缩放到224x224
    transforms.RandomHorizontalFlip(p=0.5),                # 50%概率水平翻转
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # 颜色抖动
    transforms.ToTensor()                                  # 转为张量
])

from PIL import Image
img = Image.open('图例.png')
tensor_img = augmentation_pipeline(img)
print(tensor_img.shape)  

torch.Size([4, 224, 224])


# 6.2 编程题
## 实现标签平滑后的交叉熵损失函数。

In [5]:
import torch
import torch.nn.functional as F

def label_smooth_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    """
    计算标签平滑后的交叉熵损失。
    
    参数:
        logits: 模型输出，形状 (N, C)
        labels: 真实标签，形状 (N,)
        epsilon: 平滑因子
        num_classes: 类别总数（若未给出，从logits第二维推断）
    
    返回:
        loss: 标量张量
    """
    if num_classes is None:
        num_classes = logits.size(-1)
    
    N = logits.size(0)
    # 构造平滑后的目标分布
    smooth_label = torch.full_like(logits, epsilon / (num_classes - 1))
    smooth_label.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    
    # 计算交叉熵：-sum(p_true * log(softmax(logits)))
    log_probs = F.log_softmax(logits, dim=-1)
    loss = -(smooth_label * log_probs).sum(dim=-1).mean()
    return loss

# 测试
logits = torch.randn(4, 10)  # 4个样本，10个类别
labels = torch.tensor([1, 3, 5, 7])
loss = label_smooth_cross_entropy(logits, labels, epsilon=0.1)
print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 2.4676551818847656
